# Analisi del dataset X-IIoTID

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# add X-IIoTID dataset in /data folder
df = pd.read_csv("../data/X-IIoTID dataset.csv", dtype=str, keep_default_na=False)
df.shape

(820834, 68)

## 1. Classi, livelli di etichette e valori mancanti/anomali

In [2]:
for c in ["class3", "class2", "class1"]:
    print(df[c].value_counts(), "\n")

class3
Normal    421417
Attack    399417
Name: count, dtype: int64 

class2
Normal               421417
RDOS                 141261
Reconnaissance       127590
Weaponization         67260
Lateral _movement     31596
Exfiltration          22134
Tampering              5122
C&C                    2863
Exploitation           1133
crypto-ransomware       458
Name: count, dtype: int64 

class1
Normal                            421417
RDOS                              141261
Scanning_vulnerability             52852
Generic_scanning                   50277
BruteForce                         47241
MQTT_cloud_broker_subscription     23524
Discovering_resources              23148
Exfiltration                       22134
insider_malcious                   17447
Modbus_register_reading             5953
False_data_injection                5094
C&C                                 2863
Dictionary                          2572
TCP Relay                           2119
fuzzing                            

In [3]:
# provo a convertire tutto in numero: cio' che non si converte diventa NaN
numeric_values = df.apply(pd.to_numeric, errors="coerce")
numeric_share = numeric_values.notna().mean()
numeric_cols = numeric_share[numeric_share > 0.5].index
text_cols = numeric_share[numeric_share <= 0.5].index

# valori non numerici trovati nelle colonne numeriche
non_numeric = numeric_values[numeric_cols].isna()
rows = []
for c in numeric_cols:
    for value, count in df.loc[non_numeric[c], c].value_counts().items():
        rows.append((c, repr(value), count))
summary = pd.DataFrame(rows, columns=["column", "value", "count"])
display(summary.groupby("value")["count"].sum())
display(summary.pivot_table(index="column", columns="value", values="count", aggfunc="sum", fill_value=0))

value
' '                1
'#DIV/0!'       1260
''               297
'-'          2723894
'?'              492
'aza'             18
'excel'            1
Name: count, dtype: int64

value,' ','#DIV/0!','','-','?','aza','excel'
column,,,,,,,
Avg_ideal_time,0,0,0,480,0,0,0
Avg_iowait_time,0,0,0,480,5,0,0
Avg_kbmemused,0,630,0,479,4,0,0
Avg_ldavg_1,0,0,0,526,0,0,0
Avg_nice_time,0,0,0,480,0,0,0
Avg_num_Proc/s,0,0,0,479,0,0,0
Avg_num_cswch/s,0,0,0,479,0,0,0
Avg_rtps,0,0,0,492,0,0,0
Avg_system_time,0,0,0,480,5,0,0


## 2. Feature proposte

Escludo dagli input:
- le etichette `class1`, `class2`, `class3`;
- `OSSEC_alert`, `OSSEC_alert_level`, `anomaly_alert`, perché sono segnali di altri IDS;
- data, timestamp, IP e porte, perché sono legati al testbed;
- le colonne costanti.

In [4]:
labels = ["class1", "class2", "class3"]
other_ids_alerts = ["OSSEC_alert", "OSSEC_alert_level", "anomaly_alert"]
identifiers = ["Date", "Timestamp", "Scr_IP", "Scr_port", "Des_IP", "Des_port"]
constant_cols = [c for c in df.columns if df[c].nunique() == 1]

features = [c for c in df.columns if c not in labels + other_ids_alerts + identifiers + constant_cols]
categorical_features = ["Protocol", "Service"]
numeric_features = [c for c in features if c not in categorical_features]
print("costanti:", constant_cols)
print(len(features), "feature:", features)

costanti: ['Bad_checksum', 'is_SYN_with_RST']
54 feature: ['Protocol', 'Service', 'Duration', 'Scr_bytes', 'Des_bytes', 'Conn_state', 'missed_bytes', 'is_syn_only', 'Is_SYN_ACK', 'is_pure_ack', 'is_with_payload', 'FIN or RST', 'Scr_pkts', 'Scr_ip_bytes', 'Des_pkts', 'Des_ip_bytes', 'total_bytes', 'total_packet', 'paket_rate', 'byte_rate', 'Scr_packts_ratio', 'Des_pkts_ratio', 'Scr_bytes_ratio', 'Des_bytes_ratio', 'Avg_user_time', 'Std_user_time', 'Avg_nice_time', 'Std_nice_time', 'Avg_system_time', 'Std_system_time', 'Avg_iowait_time', 'Std_iowait_time', 'Avg_ideal_time', 'Std_ideal_time', 'Avg_tps', 'Std_tps', 'Avg_rtps', 'Std_rtps', 'Avg_wtps', 'Std_wtps', 'Avg_ldavg_1', 'Std_ldavg_1', 'Avg_kbmemused', 'Std_kbmemused', 'Avg_num_Proc/s', 'Std_num_proc/s', 'Avg_num_cswch/s', 'std_num_cswch/s', 'Login_attempt', 'Succesful_login', 'File_activity', 'Process_activity', 'read_write_physical.process', 'is_privileged']


## 3. Suddivisione training/test

Le feature delle risorse dell'host sono medie e deviazioni standard su finestre di 10 secondi, quindi tutte le connessioni della stessa finestra hanno gli stessi valori. Uso questi valori come identificativo del gruppo, così una finestra finisce tutta nel training o tutta nel test.

`StratifiedGroupKFold` rispetta i gruppi e bilancia le classi di `class1`; prendo un fold (circa 20%) come test.

In [5]:
resource_cols = [c for c in df.columns if c.startswith(("Avg_", "Std_", "std_"))]
groups = df.groupby(resource_cols).ngroup()

X = df[features].replace({"TRUE": "1", "FALSE": "0", "?": None})
X[numeric_features] = X[numeric_features].apply(pd.to_numeric, errors="coerce")
y = df[labels]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(cv.split(X, df["class1"], groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(len(X_train), len(X_test))
print("finestre in comune:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
pd.DataFrame({"train": y_train["class1"].value_counts(), "test": y_test["class1"].value_counts()})

656773 164061
finestre in comune: 0


,train,test
class1,,
BruteForce,37667,9574.0
C&C,2290,573.0
Dictionary,2057,515.0
Discovering_resources,18518,4630.0
Exfiltration,17707,4427.0
Fake_notification,28,NaN
False_data_injection,4075,1019.0
Generic_scanning,40299,9978.0
MQTT_cloud_broker_subscription,18819,4705.0


Imputazione e scaling vengono adattati (`fit`) solo sul training e poi applicati al test.

In [6]:
preprocessor = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical_features),
])

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
X_train_p.shape, X_test_p.shape

((656773, 73), (164061, 73))